# 03_uncertainty_hotspots.ipynb

计算多源数据集的 Shannon Entropy，识别湿地判定争议最大的热点区域。

In [ ]:
import sys, os
sys.path.append("../src")
import xarray as xr
import geemap
import numpy as np
from wetland_analysis.data.loader import load_wetland_dataset
from wetland_analysis.analysis.uncertainty import calculate_shannon_entropy, identify_uncertainty_hotspots
from wetland_analysis.utils.geospatial import align_to_reference

# 远程代理配置
os.environ['LOCALTILESERVER_CLIENT_PREFIX'] = 'proxy/{port}'

## 1. 区域加载与多源对齐

我们将 GWD30 作为基准，将其他数据集（如 GIEMS-MC）对齐到 30m。

In [ ]:
# 1. 加载 30m 基准 (以亚马逊为例)
gwd30 = load_wetland_dataset('gwd30', year=2020)

# 2. 加载待对齐数据集
giems = load_wetland_dataset('giems_mc', year=2007) # GIEMS 时间较早，仅作演示

# 3. 空间对齐 (使用之前实现的 geospatial 工具)
giems_30m = align_to_reference(giems, gwd30, is_categorical=True)

print("Datasets aligned to 30m.")

## 2. 计算不确定性熵 (Shannon Entropy)

识别共识最低的区域。

In [ ]:
# 堆叠数据集以进行统计分析
ensemble_stack = xr.concat([gwd30, giems_30m], dim='dataset')

# 计算归一化香农熵
entropy = calculate_shannon_entropy(ensemble_stack)

m = geemap.Map()
m.add_raster(entropy, colormap='hot', name='Shannon Entropy (Disagreement)')
m.add_colorbar(colors=['#FFFF00', '#FF0000'], vmin=0, vmax=1, label='Uncertainty')
m

## 3. 提取并导出热点区域

将争议点导出为 GeoJSON，供后续遥感核查使用。

In [ ]:
hotspots = identify_uncertainty_hotspots(entropy, threshold=0.8)

# 过滤并转换为矢量 (示例代码)
# hotspots_gdf = geemap.raster_to_vector(hotspots, ...)
print("Hotspots identified. Ready for export.")